In [1]:
import pandas as pd
import numpy as np

from datasets import load_dataset


/home/maciej/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
merged_raw_df = pd.read_csv("../raw/merged_2021_2025.csv", index_col=0, header=[0,1], low_memory=False)

In [3]:
training_dataset = load_dataset("sookiemonster/asrs-narratives", split="train")
validation_dataset = load_dataset("sookiemonster/asrs-narratives", split="validation")
test_dataset = load_dataset("sookiemonster/asrs-narratives", split="test")

training_acns = set(training_dataset["acn"])
validation_acns = set(validation_dataset["acn"])
test_acns = set(test_dataset["acn"])

In [4]:
flat_headers_df = merged_raw_df.copy()
flat_headers_df.columns = [f"{main.replace(' ', '_')}_{sub.replace(' ', '_')}".lower() for main,sub in flat_headers_df.columns]
flat_headers_df.head()

,time_date,time_local_time_of_day,place_locale_reference,place_state_reference,place_relative_position.angle.radial,place_relative_position.distance.nautical_miles,place_altitude.agl.single_value,place_altitude.msl.single_value,place_latitude_/_longitude_(uas),environment_flight_conditions,...,events_detector,events_when_detected,events_result,assessments_contributing_factors_/_situations,assessments_primary_problem,report_1_narrative,report_1_callback,report_2_narrative,report_2_callback,report_1_synopsis
2260174,202507,1201-1800,BWI.Airport,MD,NaN,NaN,NaN,900.0,NaN,VMC,...,Person Air Traffic Control,In-flight,General None Reported / Taken,Procedure; Human Factors; ATC Equipment / Nav ...,Ambiguous,Aircraft vectored in within 1NM to final appro...,NaN,NaN,NaN,Air carrier Captain reported a low altitude al...
2260249,202507,0601-1200,ZZZ.Airport,US,NaN,NaN,NaN,NaN,NaN,NaN,...,Automation Aircraft Other Automation; Person F...,In-flight,Flight Crew Executed Go Around / Missed Approa...,Human Factors; Aircraft,Ambiguous,While on short final we received a glideslope ...,NaN,NaN,NaN,Air carrier pilot reported receiving an aircra...
2260370,202507,1801-2400,ZZZ.ARTCC,US,NaN,NaN,NaN,35000.0,NaN,VMC,...,Person Flight Crew,In-flight,Air Traffic Control Issued New Clearance; Air ...,Aircraft,Aircraft,Flying at cruise; FL350; the FO was the PF and...,NaN,At cruise; FL350; during level-off; the Captai...,NaN,B737 flight crew reported observing a slowing ...
2261277,202507,1801-2400,ZZZ.Airport,US,NaN,2.6,160.0,NaN,NaN,NaN,...,Person Flight Crew,In-flight,General None Reported / Taken,Human Factors,Human Factors,On Day 0 around XA:30; I forgot to get LAANC a...,Reporter stated TRUST certificate was obtained...,NaN,NaN,Recreational / Hobbyist UAS pilot reported fly...
2261317,202507,NaN,ZZZ.Airport,US,NaN,NaN,NaN,9000.0,NaN,VMC,...,Automation Aircraft Other Automation; Person F...,In-flight,Flight Crew Became Reoriented; Flight Crew Reg...,Weather; Human Factors; Procedure,Procedure,Divert into ZZZ from ZZZ1. FO flying. Vectors ...,NaN,Extremely strong winds blew us off the LOC whi...,NaN,B737 flight crew reported the pilot flying in ...


In [5]:
# To maintain consistency between events and narratives, only keep reports with ACNs in the narrative dataset
droppedCount = 0
for index, row in flat_headers_df.iterrows():
    acn = index
    if not (acn in training_acns or acn in validation_acns or acn in test_acns):
        flat_headers_df.drop(index)
        droppedCount += 1

print(droppedCount)

857


In [6]:
# Maybe could also use a Regex
columns = [
    "events_anomaly",
    "events_miss_distance",
    "events_were_passengers_involved_in_event",
    "events_detector",
    "events_when_detected",
    "events_result",
    "assessments_primary_problem"
]

# Keep only related columns
events_raw_df = flat_headers_df[columns]

In [7]:
events_raw_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25526 entries, 2260174 to 2315041
Data columns (total 7 columns):
 #   Column                                    Non-Null Count  Dtype 
---  ------                                    --------------  ----- 
 0   events_anomaly                            24982 non-null  object
 1   events_miss_distance                      3185 non-null   object
 2   events_were_passengers_involved_in_event  4146 non-null   object
 3   events_detector                           24043 non-null  object
 4   events_when_detected                      23163 non-null  object
 5   events_result                             23976 non-null  object
 6   assessments_primary_problem               25089 non-null  object
dtypes: object(7)
memory usage: 2.6+ MB


In [8]:
events_raw_df.describe()

,events_anomaly,events_miss_distance,events_were_passengers_involved_in_event,events_detector,events_when_detected,events_result,assessments_primary_problem
count,24982,3185,4146,24043,23163,23976,25089
unique,6272,513,3,378,259,5077,97
top,Aircraft Equipment Problem Critical,Vertical 200,N,Person Flight Crew,In-flight,Flight Crew Took Evasive Action,Human Factors
freq,1954,93,3896,12113,16316,2569,8370


### Normalize Event Data
We ignore NaN data for now, will be removed later

In [9]:
def ProcessEventsString(events_list, event_category, delimiter=';'):
    processed_list = []
    for event in events_list:
        if isinstance(event, str):
            event_split = event.split(delimiter)
            event_set = set(event_category + "_" + ev.lower().strip().replace(' ', '_') for ev in event_split)
            processed_list.append(list(event_set))
        else:
            processed_list.append(event)
    return processed_list

In [10]:
# When-Detected has an "other..." field that requires some extra processing
def ProcessWhenDetected(events_list):
    processed_list = []

    replacement_dict = {
        "other_" : "",
        "roll_out" : "roll",
        "rollout" : "roll",
        "take_off" : "takeoff",
        "push_back": "pushback",
        "after" : "post",
        "before" : "pre",
        "pre_flight": "preflight",
        "post_flight" : "postflight",
    }

    for event in events_list:
        if isinstance(event, str):
            event_split = event.split(';')
            event_split_processed = ["when_detected_" + ev.lower().strip().replace('-', ' ').replace(' ', '_') for ev in event_split]
            event_set = set()

            # Replace common variations of words, definitely not the most efficient approach but works for now
            for event_str in event_split_processed:
                for keyword, replacement in replacement_dict.items():
                    event_str = event_str.replace(keyword, replacement)
                event_set.add(event_str)

            processed_list.append(list(event_set))
        else:
            processed_list.append(event)

    return processed_list

In [11]:
def KeepTopKCounts(event_list, k_items_to_keep):
    unique = {}
    
    for event in event_list:
        if isinstance(event, list):
            for item in event:
                if item in unique:
                    unique[item] += 1
                else:
                    unique[item] = 1
    sorted_top_k = [k for k,v in sorted(unique.items(), reverse=True, key=lambda item: item[1])][:k_items_to_keep]
    
    output = []
    for event in event_list:
        if isinstance(event, list):
            events_in_topK = []
            for item in event:
                if item in sorted_top_k:
                    events_in_topK.append(item)
            output.append(events_in_topK)
        else:
            output.append(event)
    return output


In [12]:
# Useful for testing
def CountAndPrintUnique(event_list):
    unique = {}

    for event in event_list:
        if isinstance(event, list): # Exclude null entries
            for item in event:
                if item in unique:  
                    unique[item] += 1
                else:
                    unique[item] = 1
    print("Unique Items:", len(unique))

    df = pd.DataFrame(unique.items())
    df.rename(columns={0:"Event", 1:"Count"}, inplace=True)
    df.sort_values(by=["Count"], ascending=False, inplace=True)
    
    pd.set_option("display.max_rows", 150)
    display(df)

In [13]:
anomalies_list = ProcessEventsString(events_raw_df.events_anomaly, "anomaly")
conflict_list = ProcessEventsString(events_raw_df.events_miss_distance, "conflict")
passenger_involvement_list = ProcessEventsString(events_raw_df.events_were_passengers_involved_in_event, "passenger_involvement")
detector_list = ProcessEventsString(events_raw_df.events_detector, "detector")
result_list = ProcessEventsString(events_raw_df.events_result, "result")
#when_detected_list = ProcessEventsString(events_raw_df.events_when_detected, "when_detected")

# Originally, there were 137 "When Detected" categories
#   I arbitrarily kept 15, this likely requires further testing. This might introduce biases in the 
#   model as we could be excluding certain types of incidents/primary problems
when_detected_list = ProcessWhenDetected(events_raw_df.events_when_detected)
when_detected_list = KeepTopKCounts(when_detected_list, 15)

In [14]:
# Testing
print(len(anomalies_list) == 
      len(conflict_list) == 
      len(passenger_involvement_list) == 
      len(detector_list) == 
      len(when_detected_list) ==
      len(result_list) ==
      len(events_raw_df)
)

CountAndPrintUnique(anomalies_list)
CountAndPrintUnique(conflict_list)
CountAndPrintUnique(passenger_involvement_list)
CountAndPrintUnique(detector_list)
CountAndPrintUnique(when_detected_list)
CountAndPrintUnique(result_list)


True
Unique Items: 68


,Event,Count
3,anomaly_deviation_/_discrepancy_-_procedural_p...,14512
4,anomaly_aircraft_equipment_problem_critical,8802
15,anomaly_deviation_/_discrepancy_-_procedural_c...,5914
8,anomaly_deviation_/_discrepancy_-_procedural_far,3997
0,anomaly_atc_issue_all_types,3702
14,anomaly_conflict_nmac,2928
26,anomaly_aircraft_equipment_problem_less_severe,2920
2,anomaly_inflight_event_/_encounter_cftt_/_cfit,2705
22,anomaly_conflict_ground_conflict,2252
28,anomaly_ground_event_/_encounter_loss_of_aircr...,2216


Unique Items: 170


,Event,Count
19,conflict_vertical_0,534
7,conflict_vertical_100,486
12,conflict_horizontal_0,436
0,conflict_vertical_200,434
1,conflict_horizontal_500,338
...,...,...
165,conflict_vertical_65,1
166,conflict_horizontal_6100,1
167,conflict_horizontal_786,1
168,conflict_vertical_148,1


Unique Items: 2


...

,Event,Count
2,detector_person_flight_crew,18440
1,detector_automation_aircraft_other_automation,3859
0,detector_person_air_traffic_control,2943
8,detector_person_ground_personnel,1149
7,detector_person_flight_attendant,1015
11,detector_automation_air_traffic_control,841
10,detector_automation_aircraft_terrain_warning,736
3,detector_person_maintenance,689
4,detector_automation_aircraft_ta,431
9,detector_automation_aircraft_ra,429


Unique Items: 15


,Event,Count
0,when_detected_in_flight,17331
4,when_detected_taxi,3102
3,when_detected_aircraft_in_service_at_gate,1938
1,when_detected_routine_inspection,792
2,when_detected_preflight,577
5,when_detected_postflight,253
12,when_detected_landing,68
13,when_detected_landing_roll,55
6,when_detected_takeoff_roll,43
11,when_detected_takeoff,43


Unique Items: 36


,Event,Count
5,result_flight_crew_requested_atc_assistance_/_...,5907
9,result_general_maintenance_action,5362
13,result_flight_crew_took_evasive_action,5208
11,result_air_traffic_control_provided_assistance,5138
7,result_general_flight_cancelled_/_delayed,4738
18,result_flight_crew_overcame_equipment_problem,4376
8,result_flight_crew_landed_in_emergency_condition,3051
0,result_general_none_reported_/_taken,2729
4,result_air_traffic_control_issued_new_clearance,2501
17,result_flight_crew_returned_to_departure_airport,2460


In [15]:
# --- Conflict ---
# Decimal Distances encoded as miles (presumably)
# FAA Defines NMAC as <500 ft of distance
#CountAndPrintUnique(conflict_list)

In [16]:
# Decided to exclude miss_distance from the events list, at least for the preliminary results
# We could mess around with it later, but having numerical values like this could be difficult

combined_events = []
for i in range(len(events_raw_df)):
    combined = []
    
    if isinstance(anomalies_list[i], list): combined.extend(anomalies_list[i])
    #if isinstance(conflict_list[i], list): combined.extend(conflict_list[i])
    if isinstance(passenger_involvement_list[i], list): combined.extend(passenger_involvement_list[i])
    if isinstance(detector_list[i], list): combined.extend(detector_list[i])
    if isinstance(when_detected_list[i], list): combined.extend(when_detected_list[i])
    if isinstance(result_list[i], list): combined.extend(result_list[i])
    
    combined_events.append(combined)

events_processed_df = pd.DataFrame(combined_events, index=events_raw_df.index)
events_processed_df.head()

,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
2260174,anomaly_atc_issue_all_types,anomaly_inflight_event_/_encounter_unstabilize...,anomaly_inflight_event_/_encounter_cftt_/_cfit,anomaly_deviation_/_discrepancy_-_procedural_p...,detector_person_air_traffic_control,when_detected_in_flight,result_general_none_reported_/_taken,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2260249,anomaly_aircraft_equipment_problem_critical,anomaly_inflight_event_/_encounter_cftt_/_cfit,detector_automation_aircraft_other_automation,detector_person_flight_crew,when_detected_in_flight,result_flight_crew_became_reoriented,result_flight_crew_flc_complied_w_/_automation...,result_flight_crew_executed_go_around_/_missed...,None,None,...,None,None,None,None,None,None,None,None,None,None
2260370,anomaly_deviation_/_discrepancy_-_procedural_w...,anomaly_aircraft_equipment_problem_critical,anomaly_deviation_/_discrepancy_-_procedural_p...,detector_person_flight_crew,when_detected_in_flight,result_air_traffic_control_issued_new_clearance,result_flight_crew_requested_atc_assistance_/_...,result_aircraft_aircraft_damaged,result_general_flight_cancelled_/_delayed,result_flight_crew_landed_in_emergency_condition,...,None,None,None,None,None,None,None,None,None,None
2261277,anomaly_deviation_/_discrepancy_-_procedural_u...,anomaly_airspace_violation_all_types,anomaly_deviation_/_discrepancy_-_procedural_far,anomaly_conflict_airborne_conflict,anomaly_deviation_/_discrepancy_-_procedural_p...,detector_person_flight_crew,when_detected_in_flight,result_general_none_reported_/_taken,None,None,...,None,None,None,None,None,None,None,None,None,None
2261317,anomaly_inflight_event_/_encounter_weather_/_t...,anomaly_deviation_-_speed_all_types,anomaly_deviation_-_track_/_heading_all_types,anomaly_inflight_event_/_encounter_loss_of_air...,anomaly_deviation_/_discrepancy_-_procedural_p...,detector_automation_aircraft_other_automation,detector_person_flight_crew,when_detected_in_flight,result_flight_crew_became_reoriented,result_flight_crew_regained_aircraft_control,...,None,None,None,None,None,None,None,None,None,None


In [26]:
# Remove entries that have no events at all
for index, row in events_processed_df.iterrows():
    if row[0] == None:
        events_processed_df.drop(index=index, inplace=True)

In [27]:
events_processed_df.to_csv("../data-processing/events.csv")